# Week 5 – Model Optimization and Experimentation
## Employee Attrition Prediction Using Machine Learning

**Objective:** Optimize the baseline attrition models using systematic hyperparameter search while preserving a clean held-out test set.

This notebook implements:
- Grid Search for Logistic Regression
- Grid Search for Decision Tree
- Randomized Search for Random Forest
- 5-fold stratified cross-validation
- F1-score as the primary optimization objective
- ROC-AUC and other metrics for secondary comparison
- Baseline vs tuned-model comparison
- Reproducible experimental settings and result export

> **Important:** The test set is kept separate from hyperparameter tuning. Numerical results are generated only when this notebook is executed.


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


## 1. Load Dataset

The notebook uses the raw GitHub URL so that another evaluator can run it without depending on a local Windows path.


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/rachitgupt04/employee-attrition-prediction/main/WA_Fn-UseC_-HR-Employee-Attrition.csv"

df = pd.read_csv(DATA_URL)

print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
# Basic target preparation
df = df.copy()
df["Attrition"] = df["Attrition"].map({"No": 0, "Yes": 1})

print("Target distribution:")
print(df["Attrition"].value_counts())
print("\nMissing values:", int(df.isna().sum().sum()))


## 2. Feature Preparation

Identifier/constant columns are removed because they do not provide useful predictive information. Preprocessing is placed inside the model pipeline so that imputation, scaling and encoding are learned separately within each cross-validation fold.


In [ ]:
drop_cols = [c for c in ["EmployeeNumber", "EmployeeCount", "Over18", "StandardHours"] if c in df.columns]

X = df.drop(columns=["Attrition"] + drop_cols)
y = df["Attrition"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))


## 3. Baseline Models

The same three model families used in the earlier implementation are retained. Baseline scores provide the control condition for the optimization experiments.


In [ ]:
baseline_models = {
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "Decision Tree": Pipeline([
        ("preprocessor", preprocessor),
        ("model", DecisionTreeClassifier(random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
    ])
}

def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, proba)
    }

baseline_results = pd.DataFrame([
    evaluate_model(name, model, X_train, y_train, X_test, y_test)
    for name, model in baseline_models.items()
])

display(baseline_results.sort_values("F1", ascending=False))


## 4. Experimental Design

**Primary objective:** maximize validation F1-score because attrition prediction is a classification problem where balancing precision and recall is important.

**Controlled variables:** dataset split, random seed, preprocessing logic, five-fold stratified CV, scoring metric and held-out test set.

**Independent variables:** model hyperparameters.

**Evaluation rule:** hyperparameters are selected using training-set cross-validation only. The final test set is evaluated after the best configuration has been selected.


## 5. Experiment A – Logistic Regression Grid Search

In [ ]:
log_reg_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000, random_state=42))
])

log_reg_grid = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__class_weight": [None, "balanced"],
    "model__solver": ["liblinear"]
}

log_reg_search = GridSearchCV(
    estimator=log_reg_pipe,
    param_grid=log_reg_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

log_reg_search.fit(X_train, y_train)

print("Best CV F1:", round(log_reg_search.best_score_, 4))
print("Best parameters:", log_reg_search.best_params_)


## 6. Experiment B – Decision Tree Grid Search

In [ ]:
tree_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(random_state=42))
])

tree_grid = {
    "model__max_depth": [None, 3, 5, 7, 10],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__class_weight": [None, "balanced"]
}

tree_search = GridSearchCV(
    estimator=tree_pipe,
    param_grid=tree_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

tree_search.fit(X_train, y_train)

print("Best CV F1:", round(tree_search.best_score_, 4))
print("Best parameters:", tree_search.best_params_)


## 7. Experiment C – Random Forest Randomized Search

Randomized Search samples a controlled number of configurations rather than evaluating every possible combination. This is useful when the parameter space is larger.


In [ ]:
rf_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42, n_jobs=-1))
])

rf_distributions = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 5, 10, 15, 20],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", None],
    "model__class_weight": [None, "balanced"]
}

rf_search = RandomizedSearchCV(
    estimator=rf_pipe,
    param_distributions=rf_distributions,
    n_iter=10,
    scoring="f1",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    return_train_score=True
)

rf_search.fit(X_train, y_train)

print("Best CV F1:", round(rf_search.best_score_, 4))
print("Best parameters:", rf_search.best_params_)


## 8. Compare Tuned Models on the Held-Out Test Set

In [ ]:
searches = {
    "Logistic Regression (Tuned)": log_reg_search,
    "Decision Tree (Tuned)": tree_search,
    "Random Forest (Tuned)": rf_search
}

tuned_results = pd.DataFrame([
    evaluate_model(name, search.best_estimator_, X_train, y_train, X_test, y_test)
    for name, search in searches.items()
])

display(tuned_results.sort_values("F1", ascending=False))


## 9. Baseline vs Tuned Comparison

In [ ]:
comparison = pd.concat([
    baseline_results.assign(Stage="Baseline"),
    tuned_results.assign(Stage="Tuned")
], ignore_index=True)

display(comparison.sort_values(["Model", "Stage"]))


## 10. Cross-Validation and Generalization Analysis

A large gap between training and validation/test performance can indicate overfitting. The following table compares the best search CV F1 with final test F1.


In [ ]:
generalization_rows = []
for name, search in searches.items():
    best_model = search.best_estimator_
    best_model.fit(X_train, y_train)

    train_pred = best_model.predict(X_train)
    test_pred = best_model.predict(X_test)

    generalization_rows.append({
        "Model": name,
        "Best_CV_F1": search.best_score_,
        "Train_F1": f1_score(y_train, train_pred, zero_division=0),
        "Test_F1": f1_score(y_test, test_pred, zero_division=0),
        "CV_to_Test_F1_Gap": search.best_score_ - f1_score(y_test, test_pred, zero_division=0)
    })

generalization = pd.DataFrame(generalization_rows)
display(generalization)


## 11. Export Experiment Results

The following files are created for reproducibility and can be committed to GitHub.


In [ ]:
from pathlib import Path

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

comparison.to_csv(results_dir / "week5_model_comparison.csv", index=False)
generalization.to_csv(results_dir / "week5_generalization_analysis.csv", index=False)

with open(results_dir / "week5_best_parameters.txt", "w", encoding="utf-8") as f:
    for name, search in searches.items():
        f.write(f"{name}\n")
        f.write(f"Best CV F1: {search.best_score_:.6f}\n")
        f.write(f"Best parameters: {search.best_params_}\n\n")

print("Saved Week 5 result files in:", results_dir.resolve())


## 12. Conclusion

The optimization stage systematically searches model hyperparameters while keeping the test set untouched during tuning. The final model should be selected using the validation objective and confirmed with held-out test performance. If tuning improves validation F1 without creating a large generalization gap, the experiment provides evidence that the optimized configuration generalizes better than the baseline.

**Bayesian optimization:** This method can reduce expensive evaluations by selecting future trials based on previous results. It is discussed in the Week 5 report but is not required for this implementation because GridSearchCV and RandomizedSearchCV provide reproducible optimization using the standard scikit-learn stack.
